In [ ]:
# ===== BRANCH A (No-CoV) — CONFIG (edit ONLY this cell) ====================
#
# Constants only. The experiment knobs live in the REVIEWABLE yaml
# (experiments/stable_ac/nocov/config_nocov.yaml). On Colab that yaml sits
# inside the cloned repo, which does not exist until SETUP has cloned it and
# chdir'd to the repo root — so the yaml load happens at the top of the RUN
# cell, not here. Per-session tweaks go in OVERRIDES below.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # branch that contains experiments/stable_ac/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True            # git reset --hard so a RESTART pulls the latest push
MOUNT_DRIVE = True            # results -> Drive if True, else local ./results/

# W&B login in SETUP (whether RUN logs is the yaml's USE_WANDB, default true).
AUTHENTICATE_WANDB      = True
AUTO_AUTHENTICATE_WANDB = True   # True: promptless via Colab Secret WANDB_API_KEY;
                                 # False: prompt for the key on EVERY SETUP run.

# Merged over the yaml in RUN — e.g. {"BENCHMARK": "combined_22", "A2_MAX_WORDS": 64}
OVERRIDES = {}

print("config loaded.")

In [ ]:
# ==================== SETUP (clone / pull / install / mount) ==============
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy wandb pyyaml")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# SETUP's `git reset --hard` rewrites the .py files on disk, but Python keeps the
# OLD module objects in sys.modules for the life of the runtime -- so the RUN
# cell's `from experiments.run_baseline import run_dataset` would silently reuse
# stale code (a pull is NOT a reload). Drop them so the next import reads what
# SETUP just fetched. Without this you must Runtime -> Restart session.
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# ---- W&B authentication -------------------------------------------------
# AUTO_AUTHENTICATE_WANDB (set in CONFIG):
#   True  -> PROMPTLESS auth from a Colab Secret named WANDB_API_KEY (persists
#            across runtime restarts). Create it ONCE: Colab left sidebar ->
#            key icon (Secrets) -> add WANDB_API_KEY = <your key> -> toggle
#            'Notebook access' ON. Without a secret it reuses a valid key from
#            this session, else prompts once (a stale/invalid key is discarded,
#            not reused -- so it can always recover).
#   False -> prompt for the API key EVERY run (use to switch/refresh a key).
if AUTHENTICATE_WANDB:
    import re, wandb

    def _clean_key(k):
        # strip whitespace/newlines and stray surrounding quotes from a paste
        return (k or "").strip().strip('"').strip("'").strip()

    def _valid(k):
        return bool(re.fullmatch(r"[A-Za-z0-9_]+", k or ""))

    def _prompt_key():
        import getpass
        return _clean_key(getpass.getpass(
            "Paste your W&B API key (from wandb.ai/authorize), then Enter: "))

    _auto = AUTO_AUTHENTICATE_WANDB
    if not _auto:
        os.environ["WANDB_API_KEY"] = _prompt_key()          # always ask for a fresh key
    else:
        # auto: prefer a Colab Secret; else reuse a VALID session key; else prompt.
        _key, _why = None, ""
        try:
            from google.colab import userdata
            _key = _clean_key(userdata.get("WANDB_API_KEY"))
        except Exception as _e:
            _why = type(_e).__name__   # SecretNotFoundError / NotebookAccessError
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (promptless).")
        elif _valid(os.environ.get("WANDB_API_KEY", "")):
            print("W&B: reusing the key from this session.")
        else:
            if os.environ.get("WANDB_API_KEY"):     # stale/invalid -> drop it, don't reuse
                print("W&B: discarding an invalid key left in this session.")
                os.environ.pop("WANDB_API_KEY", None)
            print("W&B: no usable Colab Secret WANDB_API_KEY"
                  f"{(' (' + _why + ')') if _why else ''}.")
            print("     -> Promptless auth: Colab left sidebar -> key icon (Secrets)")
            print("        -> Add  name=WANDB_API_KEY  value=<key from wandb.ai/authorize>")
            print("        -> toggle 'Notebook access' ON, then re-run this cell.")
            print("     Pasting once for THIS session instead:")
            os.environ["WANDB_API_KEY"] = _prompt_key()

    # final format check before hitting the server (Colab getpass can mangle a paste)
    if not _valid(os.environ.get("WANDB_API_KEY", "")):
        print("W&B: that key still has invalid characters (allowed: A-Z a-z 0-9 _).")
        print("     Re-copy it EXACTLY from https://wandb.ai/authorize.")

    # verify; relogin=True overwrites any stale key cached in ~/.netrc
    try:
        wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True, verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        print(f"W&B: authenticated ✓  (default entity {_default}; "
              "entity/project come from config_nocov.yaml)")
    except Exception as e:
        print(f"W&B: authentication FAILED ✗ -- {e}")
        print("     tip: add a Colab Secret WANDB_API_KEY (promptless), or set "
              "AUTO_AUTHENTICATE_WANDB=False and re-run to paste a fresh key.")

In [ ]:
# ==================== RUN =================================================
# The yaml loads HERE, not in CONFIG: on Colab it lives inside the cloned
# repo, which only exists after SETUP has cloned it and chdir'd to its root.
import os
import yaml

os.environ["ACSOLVERX_ALLOW_BIG"] = "1"   # Colab runs the production budgets;
                                          # local runs must stay <= 1000 nodes.

with open("experiments/stable_ac/nocov/config_nocov.yaml") as f:
    cfg = yaml.safe_load(f)
cfg.update(OVERRIDES)
# One knob (CONFIG cell) drives both the Drive mount and the output dir.
cfg["MOUNT_DRIVE"] = MOUNT_DRIVE and IN_COLAB

from experiments.stable_ac.nocov.run_nocov import run_nocov

for budget in cfg["BUDGET"]:
    for family in cfg["FAMILIES"]:
        run_nocov(cfg, budget, family)   # resumable; one jsonl per (budget, family)